<a href="https://colab.research.google.com/github/cem8kaya/AI_voice_pipeline/blob/main/voice_AI_pipeline_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hücre 1: Sistem Kütüphaneleri ve google-genai Kurulumu

In [ ]:
# 1. Sistem bağımlılıkları (Kokoro fonem üretimi ve ses dönüşümleri için)
!apt-get update -qq && apt-get install -y -qq espeak-ng ffmpeg

# 2. Derleme araçlarını (build dependencies) PyTorch uyumlu sürüme çek
!pip install -q --upgrade pip wheel "setuptools<82"

# 3. Ana Python paketleri (Spacy sürüm çakışmasını önlemek için kısıtlamayı da aynı komuta ekliyoruz)
# NOT: WebRTC kullanılmadığı için fastrtc kaldırıldı (gradio sürüm çakışmalarına yol açıyordu). scipy: mikrofon sesini 16 kHz'e çevirmek için.
!pip install "spacy>=3.8.7,<4.0.0" faster-whisper fastembed sentence-transformers scikit-learn kokoro piper-tts google-genai soundfile numpy scipy gradio huggingface_hub pandas psutil

# Hücre 2: AI Studio Gemma Model Keşfedici, Dinamik Konfigürasyon ve Benchmark Motoru

Bu hücre; AI Studio API'sine bağlanıp hesabınızda aktif olan tüm Gemma modellerini (models.list()) otomatik çeker, konfigürasyon yapısını kurar ve RTF, TTFT, Tokens/sec, P50/P95 metriklerini hesaplayan BenchmarkTracker sınıfını hazırlar.

In [ ]:
import os
import time
import psutil
import torch
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict
from google import genai
from google.genai import types

# Colab Secrets'tan anahtarı otomatik okumayı dene
try:
    from google.colab import userdata
    DEFAULT_API_KEY = userdata.get('GEMINI_API_KEY') or ""
except Exception:
    DEFAULT_API_KEY = os.environ.get("GEMINI_API_KEY", "")

def fetch_aistudio_gemma_models(api_key: str) -> List[str]:
    """
    Google AI Studio API üzerinden mevcut tüm 'gemma' modellerini dinamik olarak listeler.
    """
    fallback_models = [
        "gemma-3-27b-it",
        "gemma-3-12b-it",
        "gemma-3-4b-it",
        "gemma-3-1b-it",
        "gemma-2-27b-it",
        "gemma-2-9b-it"
    ]
    if not api_key or len(api_key.strip()) < 10:
        return fallback_models
    try:
        client = genai.Client(api_key=api_key.strip())
        gemma_list = []
        for m in client.models.list():
            # 'models/gemma-...' formatındaki isimleri temizle ve filtrele
            name = m.name.replace("models/", "")
            actions = getattr(m, "supported_actions", None) or []
            # Sadece metin üretebilen (generateContent) Gemma modellerini listele
            if "gemma" in name.lower() and (not actions or "generateContent" in actions):
                gemma_list.append(name)
        # En yeni modelleri (gemma-3) üstte tutacak şekilde sırala
        gemma_list.sort(reverse=True)
        return gemma_list if gemma_list else fallback_models
    except Exception as e:
        print(f"⚠️ API üzerinden model listesi çekilemedi, varsayılan liste kullanılıyor: {e}")
        return fallback_models

@dataclass
class PipelineConfig:
    # --- AI Studio API & Gemma Opsiyonları ---
    gemini_api_key: str = DEFAULT_API_KEY
    llm_model_id: str = "gemma-3-27b-it"   # AI Studio'dan dinamik seçilecek Gemma modeli
    llm_temperature: float = 0.6
    llm_max_tokens: int = 120              # Görünür yanıt için bütçe
    llm_thinking_reserve: int = 1000       # Gemma 4 gibi "thinking" modeller düşünme token'larını da aynı bütçeden harcar
    chunk_min_words: int = 4               # TTS'e göndermeden önce beklenecek min. kelime sayısı

    # --- STT Opsiyonları ---
    # NOT: distil-large-v3 / *.en modelleri SADECE İngilizce bilir. Türkçe için çok dilli model gerekir.
    stt_model_id: str = "mobiuslabsgmbh/faster-whisper-large-v3-turbo"
    stt_compute_type: str = "int8_float16" # "float16", "int8_float16", "int8"
    stt_beam_size: int = 1                 # 1 = Greedy (En hızlı)
    stt_language: str = "auto"             # "auto" (her konuşmada EN/TR arasında seçer), "en", "tr"

    # --- VAD (Voice Activity Detection - Silero) Opsiyonları ---
    stt_vad_enabled: bool = True           # Kaydın sessiz kısımlarını at, konuşma yoksa STT'yi hiç çalıştırma
    stt_vad_threshold: float = 0.5         # Yüksek = daha seçici (gürültüyü konuşma sayma)
    stt_vad_min_speech_ms: int = 250       # Bundan kısa sesler konuşma sayılmaz
    stt_vad_min_silence_ms: int = 500      # Konuşma parçalarını ayırmak için gereken sessizlik
    stt_vad_speech_pad_ms: int = 300       # Konuşma parçalarının başına/sonuna eklenen tampon

    # --- Intent Router Opsiyonları ---
    intent_backend: str = "emb-minilm-multi"   # INTENT_BACKENDS anahtarı (çok dilli, Apache-2.0 / açık kaynak)
    intent_threshold: float = 0.65         # Bypass karar eşiği (motora göre değişir: UI'dan kalibre edin)
    intent_margin: float = 0.05            # 1. ve 2. intent skoru arasındaki min. fark
    enable_fast_path_cache: bool = True    # True ise LLM API'ye gitmeden RAM'den 0ms ses döner

    # --- TTS Opsiyonları ---
    tts_engine: str = "piper"              # "piper" (CPU/ONNX, EN+TR) veya "kokoro" (GPU, sadece EN)
    tts_voice: str = "af_heart"            # Kokoro sesi: "af_heart", "af_bella", "am_adam", "am_michael"
    piper_voice_en: str = "en_US-lessac-medium"   # Piper İngilizce ses profili
    piper_voice_tr: str = "tr_TR-dfki-medium"     # Piper Türkçe ses profili (Kokoro Türkçe desteklemez)
    tts_speed: float = 1.05

GLOBAL_CONFIG = PipelineConfig()

class BenchmarkTracker:
    def __init__(self):
        self.history: List[Dict] = []
        self.latest: Dict = {
            "turn_id": 0, "config_summary": "-", "transcript": "-", "response": "-",
            "intent": "-", "lang": "-", "speech_sec": 0.0, "audio_in_sec": 0.0, "stt_ms": 0.0, "stt_rtf": 0.0,
            "intent_ms": 0.0, "llm_ttft_ms": 0.0, "llm_tps": 0.0,
            "tts_first_ms": 0.0, "tts_rtf": 0.0, "total_e2e_ms": 0.0, "vram_gb": 0.0
        }

    def record_turn(self, data: Dict):
        data["turn_id"] = len(self.history) + 1
        data["vram_gb"] = round(torch.cuda.memory_allocated() / 1e9, 2) if torch.cuda.is_available() else 0.0
        self.latest = data
        self.history.append(data)

    def get_summary_df(self) -> pd.DataFrame:
        if not self.history:
            return pd.DataFrame(columns=["Turn", "Profile (STT | Gemma | TTS)", "STT (ms)", "STT RTF", "Intent (ms)", "Gemma TTFT", "Tok/s", "TTS 1st (ms)", "TTS RTF", "E2E Total (ms)"])
        rows = []
        for h in self.history[-10:]:
            rows.append({
                "Turn": h["turn_id"],
                "Profile (STT | Gemma | TTS)": h["config_summary"],
                "STT (ms)": h["stt_ms"],
                "STT RTF": f"{h['stt_rtf']:.3f}x",
                "Intent (ms)": h["intent_ms"],
                "Gemma TTFT": f"{h['llm_ttft_ms']} ms",
                "Tok/s": f"{h['llm_tps']:.1f}",
                "TTS 1st (ms)": h["tts_first_ms"],
                "TTS RTF": f"{h['tts_rtf']:.3f}x",
                "E2E Total (ms)": f"{h['total_e2e_ms']} ms"
            })
        return pd.DataFrame(rows)

    def get_percentiles(self):
        if not self.history:
            return {"p50_e2e": 0.0, "p95_e2e": 0.0}
        e2e_list = [h["total_e2e_ms"] for h in self.history]
        return {
            "p50_e2e": round(float(np.percentile(e2e_list, 50)), 1),
            "p95_e2e": round(float(np.percentile(e2e_list, 95)), 1)
        }

BENCHMARK = BenchmarkTracker()
print("AI Studio Gemma keşfedici ve Benchmark motoru hazır.")

# Hücre 3: Yerel Modeller (STT, Intent, TTS) ve AI Studio İstemci Yöneticisi (ModelRegistry)

LLM artık AI Studio üzerinden bulutta çalıştığı için yerel GPU'da sadece Distil-Whisper ve Kokoro-82M tutulur (Toplam ~2.1 GB VRAM). Ayrıca genai.Client nesnesi bağlantı havuzunu (connection pool) koruyacak şekilde önbelleklenir.

In [ ]:
import os
import ctypes

# CTranslate2 / faster-whisper için libcublas.so.12 path yaması (Namespace paketleri için düzeltildi)
try:
    import nvidia.cublas.lib
    cublas_dir = nvidia.cublas.lib.__path__[0]
    ctypes.cdll.LoadLibrary(os.path.join(cublas_dir, "libcublas.so.12"))
except Exception as e:
    print(f"⚠️ cuBLAS ön yükleme uyarısı: {e}")

try:
    import nvidia.cudnn.lib
    cudnn_dir = nvidia.cudnn.lib.__path__[0]
    ctypes.cdll.LoadLibrary(os.path.join(cudnn_dir, "libcudnn.so.9"))
except Exception as e:
    pass

import gc
import wave
import io
import numpy as np
import torch
from faster_whisper import WhisperModel
from kokoro import KPipeline
from piper.voice import PiperVoice
from huggingface_hub import hf_hub_download

# Piper ses profilleri: ad -> HuggingFace (rhasspy/piper-voices) yolu ve dili
PIPER_VOICES = {
    "en_US-lessac-medium": ("en/en_US/lessac/medium/en_US-lessac-medium", "en"),
    "en_US-amy-medium":    ("en/en_US/amy/medium/en_US-amy-medium", "en"),
    "en_US-ryan-medium":   ("en/en_US/ryan/medium/en_US-ryan-medium", "en"),
    "en_US-joe-medium":    ("en/en_US/joe/medium/en_US-joe-medium", "en"),
    "tr_TR-dfki-medium":       ("tr/tr_TR/dfki/medium/tr_TR-dfki-medium", "tr"),
    "tr_TR-fahrettin-medium":  ("tr/tr_TR/fahrettin/medium/tr_TR-fahrettin-medium", "tr"),
    "tr_TR-fettah-medium":     ("tr/tr_TR/fettah/medium/tr_TR-fettah-medium", "tr"),
}
KOKORO_VOICES = ["af_heart", "af_bella", "am_adam", "am_michael"]

def piper_voice_choices(lang: str):
    return [name for name, (_, l) in PIPER_VOICES.items() if l == lang]

def resolve_tts(cfg, lang: str):
    """Konuşma diline göre (motor, ses) seç. Kokoro Türkçe bilmediği için Türkçe her zaman Piper'a gider."""
    if lang == "tr":
        return "piper", cfg.piper_voice_tr
    if cfg.tts_engine == "kokoro":
        return "kokoro", cfg.tts_voice
    return "piper", cfg.piper_voice_en

def to_float32_numpy(audio) -> np.ndarray:
    """Kokoro torch.Tensor (GPU/CPU) döndürür; Gradio/numpy ise ndarray ister."""
    if audio is None:
        return np.zeros(0, dtype=np.float32)
    if hasattr(audio, "detach"):          # torch.Tensor
        audio = audio.detach().cpu().numpy()
    return np.asarray(audio, dtype=np.float32).reshape(-1)

class ModelRegistry:
    def __init__(self):
        self.stt_model = None
        self.current_stt_key = None

        self.genai_client = None
        self.current_api_key = None

        self.kokoro_tts = None
        self.piper_voices = {}   # ses adı -> PiperVoice

    def get_genai_client(self, api_key: str) -> genai.Client:
        clean_key = api_key.strip()
        if not clean_key:
            raise ValueError("Lütfen sol panelden geçerli bir Google AI Studio (GEMINI_API_KEY) anahtarı girin!")
        if self.genai_client is None or self.current_api_key != clean_key:
            self.genai_client = genai.Client(api_key=clean_key)
            self.current_api_key = clean_key
        return self.genai_client

    def get_stt(self, model_id: str, compute_type: str) -> WhisperModel:
        key = f"{model_id}_{compute_type}"
        if self.current_stt_key != key:
            print(f"🔄 [STT Yükleniyor] {key}...")
            if self.stt_model is not None:
                del self.stt_model
                gc.collect()
                torch.cuda.empty_cache()
            if torch.cuda.is_available():
                device, ctype = "cuda", compute_type
            else:
                print("⚠️ GPU bulunamadı (Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU). CPU/int8 kullanılıyor, yavaş olacak.")
                device, ctype = "cpu", "int8"
            self.stt_model = WhisperModel(model_id, device=device, compute_type=ctype)
            self.current_stt_key = key
            _ = list(self.stt_model.transcribe(np.zeros(16000, dtype=np.float32), beam_size=1)[0])
        return self.stt_model

    def synthesize_tts(self, text: str, engine: str, voice: str, speed: float):
        """(sample_rate, float32 numpy) parçaları üretir."""
        if not text or not text.strip():
            return
        if engine == "kokoro":
            if self.kokoro_tts is None:
                print("🔄 Kokoro-82M yükleniyor...")
                self.kokoro_tts = KPipeline(lang_code='a')
                _ = [to_float32_numpy(a) for _, _, a in self.kokoro_tts("Warm up.", voice='af_heart', speed=1.0)]
            for _, _, audio_chunk in self.kokoro_tts(text, voice=voice, speed=speed):
                audio_np = to_float32_numpy(audio_chunk)
                if audio_np.size:
                    yield (24000, audio_np)

        elif engine == "piper":
            if voice not in PIPER_VOICES:
                raise ValueError(f"Bilinmeyen Piper sesi: {voice}")
            if voice not in self.piper_voices:
                print(f"🔄 Piper sesi indiriliyor ve yükleniyor: {voice}...")
                base = PIPER_VOICES[voice][0]
                onnx_path = hf_hub_download(repo_id="rhasspy/piper-voices", filename=f"{base}.onnx")
                json_path = hf_hub_download(repo_id="rhasspy/piper-voices", filename=f"{base}.onnx.json")
                self.piper_voices[voice] = PiperVoice.load(onnx_path, config_path=json_path, use_cuda=False)
            piper_voice = self.piper_voices[voice]

            length_scale = 1.0 / max(speed, 0.1)
            try:
                # piper-tts >= 1.3: synthesize(text, syn_config) -> AudioChunk iterator
                from piper import SynthesisConfig
                syn_cfg = SynthesisConfig(length_scale=length_scale)
                for ch in piper_voice.synthesize(text, syn_config=syn_cfg):
                    audio_np = to_float32_numpy(ch.audio_float_array)
                    if audio_np.size:
                        yield (ch.sample_rate, audio_np)
            except ImportError:
                # piper-tts < 1.3: synthesize(text, wav_file, length_scale=...)
                wav_io = io.BytesIO()
                with wave.open(wav_io, "wb") as wav_file:
                    piper_voice.synthesize(text, wav_file, length_scale=length_scale)
                wav_io.seek(0)
                with wave.open(wav_io, "rb") as wav_read:
                    sr = wav_read.getframerate()
                    frames = wav_read.readframes(wav_read.getnframes())
                    yield (sr, np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0)
        else:
            raise ValueError(f"Bilinmeyen TTS motoru: {engine}")

REGISTRY = ModelRegistry()

print("Yerel STT ve TTS modelleri ısıtılıyor...")
REGISTRY.get_stt(GLOBAL_CONFIG.stt_model_id, GLOBAL_CONFIG.stt_compute_type)
for _lang, _txt in (("en", "System initialized."), ("tr", "Sistem hazır.")):
    _eng, _voice = resolve_tts(GLOBAL_CONFIG, _lang)
    _ = list(REGISTRY.synthesize_tts(_txt, _eng, _voice, 1.0))
print(f"✅ Hazır! Yerel GPU VRAM Tüketimi: {torch.cuda.memory_allocated() / 1e9:.2f} GB (LLM AI Studio üzerinde çalışacak)")

# Hücre 3.5: Çok Dilli (TR+EN) Intent Router Çerçevesi
Seçilebilir, açık kaynak motorlar (Apache-2.0 modeller / BSD-3 scikit-learn) + eşik/margin kalibrasyonu. Kendi router'ınızı `IntentBackend` sınıfını genişletip `register_intent_backend(...)` ile ekleyebilirsiniz; kendi intent'leriniz için `DEFAULT_INTENTS` listesini düzenleyin veya `ROUTER.set_intents(...)` kullanın.

In [ ]:
import abc
import re
import time
import unicodedata
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

OOD_INTENT = "general_llm_chat"   # Hiçbir intent eşleşmezse LLM'e gidilir

# =====================================================================================
# 1) INTENT TANIMLARI (çok dilli): kendi intent'lerinizi buraya ekleyin / değiştirin.
#    examples  -> dil -> örnek cümleler (router'ın "eğitim" verisi)
#    responses -> dil -> fast-path'te LLM'e gitmeden seslendirilecek hazır yanıt
# =====================================================================================
@dataclass
class Intent:
    id: str
    examples: Dict[str, List[str]]
    responses: Dict[str, str] = field(default_factory=dict)
    description: str = ""

DEFAULT_INTENTS: List[Intent] = [
    Intent(
        id="check_order_status",
        description="Sipariş / kargo durumu sorgulama",
        examples={
            "en": ["Where is my order?", "Track my package", "When will my shipment arrive?",
                   "Can you check my delivery status?", "My package is late",
                   "Has my order shipped yet?", "I want to know where my parcel is"],
            "tr": ["Siparişim nerede?", "Kargom ne zaman gelecek?", "Paketimi takip etmek istiyorum",
                   "Siparişimin durumunu öğrenebilir miyim?", "Kargom geç kaldı",
                   "Siparişim kargoya verildi mi?", "Gönderim durumunu kontrol eder misin?",
                   "Paketim hâlâ gelmedi"],
        },
        responses={
            "en": "I just checked your account. Your order is currently in transit and scheduled for delivery tomorrow by 5 PM.",
            "tr": "Hesabınızı kontrol ettim. Siparişiniz şu anda yolda ve yarın saat beşe kadar teslim edilecek.",
        },
    ),
    Intent(
        id="cancel_order",
        description="Sipariş iptali",
        examples={
            "en": ["I want to cancel my order", "Cancel my purchase",
                   "Please cancel the order I placed yesterday", "I changed my mind, I don't want it anymore",
                   "Stop my order from shipping"],
            "tr": ["Siparişimi iptal etmek istiyorum", "Satın aldığımı iptal et",
                   "Dün verdiğim siparişi iptal eder misin?", "Vazgeçtim, artık istemiyorum",
                   "Siparişin gönderilmesini durdur"],
        },
        responses={
            "en": "I can help you cancel it. Could you please tell me your order number?",
            "tr": "Siparişinizi iptal etmenize yardımcı olabilirim. Lütfen sipariş numaranızı söyler misiniz?",
        },
    ),
    Intent(
        id="book_appointment",
        description="Randevu / toplantı oluşturma",
        examples={
            "en": ["I want to schedule a meeting", "Book an appointment", "Can we set up a call?",
                   "Are you available tomorrow?", "Schedule a consultation",
                   "I would like to make a reservation for next week"],
            "tr": ["Bir toplantı ayarlamak istiyorum", "Randevu almak istiyorum",
                   "Bir görüşme planlayabilir miyiz?", "Yarın müsait misiniz?",
                   "Danışmanlık için randevu oluştur", "Gelecek hafta için bir saat ayırtmak istiyorum",
                   "Sizinle görüşme ayarlayalım"],
        },
        responses={
            "en": "I would love to help schedule that. I have open slots tomorrow at 10 AM and 2 PM Eastern. Which one works best for you?",
            "tr": "Memnuniyetle ayarlayayım. Yarın saat on ve on dörtte boş yerim var. Hangisi size uygun?",
        },
    ),
    Intent(
        id="pricing_inquiry",
        description="Fiyat / abonelik bilgisi",
        examples={
            "en": ["How much does this cost?", "What are your pricing plans?", "Tell me about the subscription fee",
                   "Is there a free trial?", "What is the price for enterprise?", "How much is the monthly plan?"],
            "tr": ["Bu ne kadar tutuyor?", "Fiyatlandırma planlarınız neler?", "Abonelik ücreti ne kadar?",
                   "Ücretsiz deneme var mı?", "Kurumsal paketin fiyatı nedir?", "Aylık plan kaç lira?"],
        },
        responses={
            "en": "Our open-source enterprise pipeline uses free Gemma models via AI Studio, reducing your operational cost by over ninety five percent.",
            "tr": "Açık kaynaklı kurumsal altyapımız AI Studio üzerindeki ücretsiz Gemma modellerini kullanır ve işletme maliyetinizi yüzde doksan beşin üzerinde azaltır.",
        },
    ),
    Intent(
        id="greeting",
        description="Selamlaşma",
        examples={
            "en": ["Hello", "Hi there", "Good morning", "Hey, how are you?", "Good evening"],
            "tr": ["Merhaba", "Selam", "Günaydın", "İyi akşamlar", "Nasılsın?"],
        },
        responses={
            "en": "Hello! How can I help you today?",
            "tr": "Merhaba! Size nasıl yardımcı olabilirim?",
        },
    ),
]

# Eşik/margin kalibrasyonu ve regresyon testi için etiketli test cümleleri (örneklerden FARKLI).
# (metin, dil, beklenen_intent)  — beklenen OOD_INTENT ise router fast-path'e girmemeli.
# Kendi alanınıza göre bu listeyi büyütün; ne kadar çok o kadar güvenilir kalibrasyon.
INTENT_TEST_SET: List[Tuple[str, str, str]] = [
    ("Where's my parcel right now?", "en", "check_order_status"),
    ("Has the package left the warehouse?", "en", "check_order_status"),
    ("I'd like to book a time with you", "en", "book_appointment"),
    ("Do you have any slots next Tuesday?", "en", "book_appointment"),
    ("What does the premium plan cost?", "en", "pricing_inquiry"),
    ("Is it free to try?", "en", "pricing_inquiry"),
    ("Please cancel my order", "en", "cancel_order"),
    ("I don't want the item anymore, cancel it", "en", "cancel_order"),
    ("Hi!", "en", "greeting"),
    ("Good afternoon", "en", "greeting"),
    ("Tell me a joke", "en", OOD_INTENT),
    ("What is the capital of France?", "en", OOD_INTENT),
    ("How do I bake a cake?", "en", OOD_INTENT),
    ("Explain quantum computing", "en", OOD_INTENT),
    ("Define yourself", "en", OOD_INTENT),
    ("Kargom şu an nerede?", "tr", "check_order_status"),
    ("Paket depodan çıktı mı?", "tr", "check_order_status"),
    ("Seninle bir saat ayarlamak istiyorum", "tr", "book_appointment"),
    ("Önümüzdeki salı için boş yeriniz var mı?", "tr", "book_appointment"),
    ("Premium paket ne kadar?", "tr", "pricing_inquiry"),
    ("Denemek ücretsiz mi?", "tr", "pricing_inquiry"),
    ("Lütfen siparişimi iptal edin", "tr", "cancel_order"),
    ("Ürünü artık istemiyorum, iptal edelim", "tr", "cancel_order"),
    ("Merhaba, kolay gelsin", "tr", "greeting"),
    ("İyi günler", "tr", "greeting"),
    ("Bana bir fıkra anlat", "tr", OOD_INTENT),
    ("Fransa'nın başkenti neresi?", "tr", OOD_INTENT),
    ("Kek nasıl yapılır?", "tr", OOD_INTENT),
    ("Kuantum bilgisayarlar nasıl çalışır?", "tr", OOD_INTENT),
    ("Bugün hava nasıl olacak?", "tr", OOD_INTENT),
    ("Kendini tanıt", "tr", OOD_INTENT),
]

# =====================================================================================
# 2) METİN NORMALİZASYONU (Türkçe'ye duyarlı)
# =====================================================================================
_TR_FOLD = str.maketrans({"ç": "c", "ğ": "g", "ı": "i", "ö": "o", "ş": "s", "ü": "u",
                          "â": "a", "î": "i", "û": "u"})

def normalize_text(text: str, lang: str = "en", fold_diacritics: bool = False) -> str:
    """
    - Unicode NFC, boşluk sadeleştirme
    - Türkçe'de İ/I -> i/ı (Python'un .lower()'ı 'I'yı 'i' yapar, bu Türkçe'de yanlış)
    - fold_diacritics=True: ç,ğ,ı,ö,ş,ü -> c,g,i,o,s,u ve noktalama kaldırma (lexical backend'ler için;
      STT/klavye kaynaklı aksan farklarına dayanıklı olur)
    """
    t = unicodedata.normalize("NFC", text or "").strip()
    if lang == "tr":
        t = t.replace("İ", "i").replace("I", "ı")
    t = t.lower()
    if fold_diacritics:
        t = t.translate(_TR_FOLD)
        t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

# =====================================================================================
# 3) BACKEND ARAYÜZÜ + KAYIT DEFTERİ (kendi router'ınızı buradan takarsınız)
# =====================================================================================
class IntentBackend(abc.ABC):
    """Bir intent sınıflandırma motoru. Kendi yönteminizi bu sınıfı genişleterek yazın."""
    @abc.abstractmethod
    def fit(self, intents: List[Intent]) -> None:
        """Intent örneklerinden indeks/model hazırla."""
    @abc.abstractmethod
    def scores(self, text: str, lang: str) -> Dict[str, float]:
        """Her intent id'si için benzerlik/güven skoru döndür (yüksek = daha uygun, ~0..1)."""

@dataclass
class BackendSpec:
    key: str
    label: str
    factory: Callable[[], IntentBackend]
    default_threshold: float = 0.65
    default_margin: float = 0.05
    license: str = ""
    description: str = ""

INTENT_BACKENDS: Dict[str, BackendSpec] = {}

def register_intent_backend(key, label, factory, default_threshold=0.65, default_margin=0.05,
                            license="", description=""):
    """Yeni bir router motorunu kaydeder. UI hücresini çalıştırmadan ÖNCE çağırın; dropdown'da görünür."""
    INTENT_BACKENDS[key] = BackendSpec(key, label, factory, default_threshold, default_margin, license, description)

def _flatten_examples(intents: List[Intent]):
    texts, labels, langs = [], [], []
    for it in intents:
        for lang, phrases in it.examples.items():
            for p in phrases:
                texts.append(p); labels.append(it.id); langs.append(lang)
    return texts, labels, langs

def _max_per_intent(sims: np.ndarray, labels: List[str]) -> Dict[str, float]:
    out: Dict[str, float] = {}
    for s, lab in zip(sims, labels):
        if s > out.get(lab, -1.0):
            out[lab] = float(s)
    return out

# ---------------- Backend A: Çok dilli embedding (fastembed ONNX veya sentence-transformers) ----------------
class EmbeddingIntentBackend(IntentBackend):
    """
    Çok dilli cümle gömmeleri + kosinüs benzerliği (intent başına en yakın örnek).
    Aynı vektör uzayında TR/EN karışık örnekler kullanılır (cross-lingual).
    library: "auto" (fastembed destekliyorsa onu, yoksa sentence-transformers), "fastembed", "st"
    """
    def __init__(self, model_id: str, library: str = "auto"):
        self.model_id, self.library = model_id, library
        self._fe = self._st = None
        self.X: Optional[np.ndarray] = None
        self.labels: List[str] = []

    def _load(self):
        if self._fe is not None or self._st is not None:
            return
        use_fe = self.library == "fastembed"
        if self.library == "auto":
            try:
                from fastembed import TextEmbedding
                use_fe = self.model_id in {m["model"] for m in TextEmbedding.list_supported_models()}
            except Exception:
                use_fe = False
        print(f"🔄 [Intent Router] {self.model_id} yükleniyor ({'fastembed' if use_fe else 'sentence-transformers'})...")
        if use_fe:
            from fastembed import TextEmbedding
            self._fe = TextEmbedding(model_name=self.model_id)
        else:
            from sentence_transformers import SentenceTransformer
            device = "cuda" if torch.cuda.is_available() else "cpu"
            self._st = SentenceTransformer(self.model_id, device=device)

    def _embed(self, texts: List[str]) -> np.ndarray:
        self._load()
        if self._fe is not None:
            vecs = np.asarray(list(self._fe.embed(texts)), dtype=np.float32)
        else:
            vecs = np.asarray(self._st.encode(texts, convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)
        return vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-9)

    def fit(self, intents):
        texts, self.labels, langs = _flatten_examples(intents)
        self.X = self._embed([normalize_text(t, l) for t, l in zip(texts, langs)])
        _ = self._embed(["warmup"])

    def scores(self, text, lang):
        q = self._embed([normalize_text(text, lang)])[0]
        return _max_per_intent(self.X @ q, self.labels)

# ---------------- Backend B: TF-IDF karakter n-gram (model indirmez, çok hızlı, Türkçe eklerine dayanıklı) ----------------
class TfidfCharIntentBackend(IntentBackend):
    """scikit-learn TF-IDF (char_wb 2-5 gram). Ekli/çekimli Türkçe kelimelerde ve yazım farklarında iyi çalışır;
    anlamsal (eş anlamlı) genelleme yapmaz — embedding ile birlikte (hybrid) kullanmak mantıklıdır."""
    def __init__(self):
        self.vec = None
        self.X = None
        self.labels: List[str] = []

    def fit(self, intents):
        from sklearn.feature_extraction.text import TfidfVectorizer
        texts, self.labels, langs = _flatten_examples(intents)
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)
        self.X = self.vec.fit_transform([normalize_text(t, l, fold_diacritics=True) for t, l in zip(texts, langs)])

    def scores(self, text, lang):
        q = self.vec.transform([normalize_text(text, lang, fold_diacritics=True)])
        sims = (self.X @ q.T).toarray().ravel()
        return _max_per_intent(sims, self.labels)

# ---------------- Backend C: Hibrit (embedding + TF-IDF ağırlıklı toplam) ----------------
class HybridIntentBackend(IntentBackend):
    def __init__(self, emb: IntentBackend, lex: IntentBackend, w_emb: float = 0.7):
        self.emb, self.lex, self.w = emb, lex, w_emb

    def fit(self, intents):
        self.emb.fit(intents); self.lex.fit(intents)

    def scores(self, text, lang):
        a, b = self.emb.scores(text, lang), self.lex.scores(text, lang)
        return {k: self.w * a.get(k, 0.0) + (1 - self.w) * b.get(k, 0.0) for k in set(a) | set(b)}

# ---- Hazır seçenekler (hepsi açık kaynak; Apache-2.0 modeller + BSD-3 scikit-learn) ----
register_intent_backend(
    "emb-minilm-multi", "Embedding • paraphrase-multilingual-MiniLM-L12-v2 (50+ dil, TR ✓, ~220 MB, en hızlı)",
    lambda: EmbeddingIntentBackend("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"),
    default_threshold=0.65, default_margin=0.05, license="Apache-2.0",
    description="Varsayılan. Küçük ve hızlı; kısa komut/niyet cümlelerinde iyi bir başlangıç.")
register_intent_backend(
    "emb-mpnet-multi", "Embedding • paraphrase-multilingual-mpnet-base-v2 (TR ✓, ~1 GB, daha doğru)",
    lambda: EmbeddingIntentBackend("sentence-transformers/paraphrase-multilingual-mpnet-base-v2"),
    default_threshold=0.65, default_margin=0.05, license="Apache-2.0",
    description="MiniLM'den daha büyük ve genelde daha isabetli; gecikme biraz daha yüksek.")
register_intent_backend(
    "emb-qwen3-q", "Embedding • Qwen3-Embedding-0.6B (int8) (100+ dil, TR ✓, ~1.1 GB, en güçlü)",
    lambda: EmbeddingIntentBackend("Qwen/Qwen3-Embedding-0.6B-Q", library="fastembed"),
    default_threshold=0.70, default_margin=0.05, license="Apache-2.0",
    description="Daha yeni/güçlü çok dilli model (fastembed >= 0.7.4 gerekir). Alakasız cümlelerde bile benzerlik tabanı yüksek olabilir: eşiği kalibre edin.")
register_intent_backend(
    "emb-labse", "Embedding • LaBSE (109 dil, TR ✓, ~1.8 GB, sentence-transformers)",
    lambda: EmbeddingIntentBackend("sentence-transformers/LaBSE", library="st"),
    default_threshold=0.65, default_margin=0.05, license="Apache-2.0",
    description="Çeviri eşleştirme odaklı; dillerarası (TR↔EN) eşleşmede güçlü, aynı dilde paraphrase'de MiniLM/mpnet kadar iyi olmayabilir.")
register_intent_backend(
    "tfidf-char", "Sözcüksel • TF-IDF karakter n-gram (model yok, anlık, TR ekleri için dayanıklı)",
    lambda: TfidfCharIntentBackend(),
    default_threshold=0.50, default_margin=0.05, license="BSD-3 (scikit-learn)",
    description="Hiç model indirmez. Eş anlamlıları anlamaz; örnek cümlelere yakın ifadelerde çok hızlı ve açıklanabilir.")
register_intent_backend(
    "hybrid-minilm-tfidf", "Hibrit • MiniLM (çok dilli) + TF-IDF (0.7 / 0.3)",
    lambda: HybridIntentBackend(EmbeddingIntentBackend("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"),
                                TfidfCharIntentBackend(), 0.7),
    default_threshold=0.58, default_margin=0.05, license="Apache-2.0 + BSD-3",
    description="Anlamsal benzerlik + yazım benzerliği; tek başına embedding'in kaçırdığı kısa/ekli ifadelerde dengeleyici.")

# =====================================================================================
# 4) ROUTER (backend'den bağımsız karar mantığı + fast-path ses önbelleği)
# =====================================================================================
@dataclass
class RouteResult:
    intent: str                     # eşleşme yoksa OOD_INTENT
    matched: bool
    score: float
    margin: float                   # 1. ve 2. intent skoru farkı
    ranked: List[Tuple[str, float]]
    response_text: Optional[str]
    audio: Optional[tuple]
    latency_ms: float
    backend: str

class IntentRouter:
    def __init__(self, intents: List[Intent]):
        self.intents = list(intents)
        self.backend: Optional[IntentBackend] = None
        self.backend_key: Optional[str] = None
        self.audio_cache: Dict[tuple, tuple] = {}

    # --- yaşam döngüsü ---
    def activate(self, key: str) -> None:
        """Seçilen backend'i oluştur ve intent örnekleriyle indeksle."""
        if key not in INTENT_BACKENDS:
            raise ValueError(f"Bilinmeyen intent backend'i: {key} (kayıtlı: {list(INTENT_BACKENDS)})")
        if self.backend is not None and self.backend_key == key:
            return
        t0 = time.perf_counter()
        backend = INTENT_BACKENDS[key].factory()
        backend.fit(self.intents)
        self.backend, self.backend_key = backend, key
        print(f"✅ [Intent Router] '{key}' hazır ({(time.perf_counter() - t0):.1f} sn).")

    def set_intents(self, intents: List[Intent]) -> None:
        """Intent listesini değiştir (kendi intent'leriniz) ve aktif backend'i yeniden indeksle."""
        self.intents = list(intents)
        self.audio_cache.clear()
        if self.backend is not None:
            self.backend.fit(self.intents)

    def response_for(self, intent_id: str, lang: str) -> Optional[str]:
        for it in self.intents:
            if it.id == intent_id:
                return it.responses.get(lang) or it.responses.get("en")
        return None

    # --- fast-path ses önbelleği (dil + TTS profiline göre) ---
    def _audio_key(self, cfg, lang, intent_id):
        eng, voice = resolve_tts(cfg, lang)
        return (eng, voice, round(float(cfg.tts_speed), 3), lang, intent_id)

    def prepare_audio(self, cfg, langs=("en", "tr")) -> None:
        for lang in langs:
            for it in self.intents:
                text = self.response_for(it.id, lang)
                if not text or (lang not in it.responses):
                    continue
                key = self._audio_key(cfg, lang, it.id)
                if key in self.audio_cache:
                    continue
                try:
                    chunks, sr_out = [], 24000
                    eng, voice = resolve_tts(cfg, lang)
                    for sr, audio in REGISTRY.synthesize_tts(text, eng, voice, cfg.tts_speed):
                        sr_out = sr; chunks.append(audio)
                    if chunks:
                        self.audio_cache[key] = (sr_out, np.concatenate(chunks))
                except Exception as e:
                    print(f"⚠️ Fast-path sesi hazırlanamadı ({lang}/{it.id}): {e}")

    # --- sınıflandırma ---
    def rank(self, text: str, lang: str) -> List[Tuple[str, float]]:
        if self.backend is None:
            raise RuntimeError("Intent router henüz etkinleştirilmedi (ROUTER.activate).")
        sc = self.backend.scores(text, lang)
        return sorted(sc.items(), key=lambda kv: kv[1], reverse=True)

    def decide(self, ranked, threshold: float, margin_min: float):
        s1 = ranked[0][1] if ranked else 0.0
        s2 = ranked[1][1] if len(ranked) > 1 else 0.0
        margin = s1 - s2
        ok = bool(ranked) and s1 >= threshold and margin >= margin_min
        return (ranked[0][0] if ok else OOD_INTENT), ok, s1, margin

    def route(self, text: str, lang: str, cfg) -> RouteResult:
        t0 = time.perf_counter()
        ranked = self.rank(text, lang)
        latency_ms = (time.perf_counter() - t0) * 1000
        intent, ok, s1, margin = self.decide(ranked, cfg.intent_threshold, cfg.intent_margin)
        resp_text = audio = None
        if ok:
            resp_text = self.response_for(intent, lang)
            key = self._audio_key(cfg, lang, intent)
            if key not in self.audio_cache:
                self.prepare_audio(cfg, langs=(lang,))
            audio = self.audio_cache.get(key)
        return RouteResult(intent, ok, s1, margin, ranked, resp_text, audio, latency_ms, self.backend_key or "-")

ROUTER = IntentRouter(DEFAULT_INTENTS)

# =====================================================================================
# 5) DEĞERLENDİRME / KALİBRASYON YARDIMCILARI
# =====================================================================================
def evaluate_intent_router(router: IntentRouter, test_set=None, margin_min: float = 0.05, thresholds=None):
    """
    Test cümlelerini aktif backend ile skorlar ve eşik taraması yapar.
    wrong_fast = yanlış intent'e fast-path (en pahalı hata); fallback = LLM'e düşen doğru-intent cümleleri.
    Önerilen eşik: wrong_fast oranı <= %3 olan eşikler arasında en yüksek doğruluk (eşitlikte daha yüksek eşik).
    """
    test_set = test_set or INTENT_TEST_SET
    thresholds = thresholds if thresholds is not None else [round(x, 2) for x in np.arange(0.30, 0.96, 0.05)]
    ranked_all = [router.rank(t, l) for t, l, _ in test_set]
    rows = []
    for thr in thresholds:
        correct = wrong_fast = fallback = 0
        for ranked, (_, _, exp) in zip(ranked_all, test_set):
            pred, ok, _, _ = router.decide(ranked, thr, margin_min)
            correct += int(pred == exp)
            wrong_fast += int(ok and pred != exp)
            fallback += int((not ok) and exp != OOD_INTENT)
        n = len(test_set)
        rows.append({"threshold": thr, "accuracy": correct / n, "wrong_fast": wrong_fast / n, "missed_to_llm": fallback / n})
    safe = [r for r in rows if r["wrong_fast"] <= 0.03] or [min(rows, key=lambda r: r["wrong_fast"])]
    best = max(safe, key=lambda r: (r["accuracy"], r["threshold"]))
    return best, pd.DataFrame(rows), ranked_all

print("✅ Intent Router çerçevesi hazır. Kayıtlı motorlar:", ", ".join(INTENT_BACKENDS))
try:
    ROUTER.activate(GLOBAL_CONFIG.intent_backend)
    ROUTER.prepare_audio(GLOBAL_CONFIG)
except Exception as _e:   # indirme/ortam hatası tüm notebook'u durdurmasın; UI'dan başka motor seçilebilir
    print(f"⚠️ Varsayılan intent router etkinleştirilemedi ({GLOBAL_CONFIG.intent_backend}): {_e}")

# ---------------------------------------------------------------------------------
# KENDİ ROUTER'INIZI EKLEMEK İÇİN ÖRNEK (yorumu kaldırıp UI hücresinden önce çalıştırın):
#
# class MyIntentBackend(IntentBackend):
#     def fit(self, intents):
#         texts, self.labels, _ = _flatten_examples(intents)   # kendi modelinizi/indeksinizi burada hazırlayın
#     def scores(self, text, lang):
#         return {"check_order_status": 0.9, "pricing_inquiry": 0.1}   # intent_id -> skor
#
# register_intent_backend("my-router", "Benim Router'ım", lambda: MyIntentBackend(),
#                         default_threshold=0.6, license="Apache-2.0")
# ROUTER.activate("my-router")
# ---------------------------------------------------------------------------------

# Hücre 4: Semantic Intent Router ve AI Studio Gemma Streaming Orkestratörü
Mühendislik Detayı (Gemma Sistem Prompt Uyumluluğu):

AI Studio üzerindeki bazı Gemma modelleri (gemma-2 ve erken gemma-3 sürümleri) ayrı bir system_instruction alanını reddedip sadece user ve model rollerini kabul edebilir. Bu yüzden aşağıdaki kodda sistem talimatını konuşma geçmişinin ilk mesajına güvenli bir şekilde gömüyoruz; böylece hangi Gemma modelini seçerseniz seçin %100 sıfır API hatasıyla (generate_content_stream) akış sağlanır.

In [ ]:
import re
from math import gcd

SYSTEM_INSTRUCTION = (
    "You are a fast, concise, and helpful AI voice assistant. "
    "Always answer in 1 to 2 short, natural conversational sentences suitable for text-to-speech. "
    "Never use markdown, bullet points, asterisks, or emojis. "
    "Always reply in the same language the user speaks (English or Turkish)."
)
LANG_NAMES = {"en": "English", "tr": "Turkish"}
# Her elemanı {"role": "user" | "model", "text": "..."} formatında tutuyoruz
CHAT_HISTORY: List[Dict[str, str]] = []

def build_gemma_contents(history: List[Dict[str, str]], latest_user_text: str, lang: str = "en") -> List[types.Content]:
    """
    Tüm Gemma versiyonlarıyla (Gemma 2 & Gemma 3) %100 uyumlu olması için
    sistem yönergesini ilk 'user' mesajına enjekte eder ve rolleri ('user'/'model') dengeler.
    """
    recent = history[-6:] # Son 6 mesajı tut (düşük gecikme için)
    contents = []

    for idx, msg in enumerate(recent):
        text_val = msg["text"]
        if idx == 0 and msg["role"] == "user":
            text_val = f"[System Instructions: {SYSTEM_INSTRUCTION}]\n\nUser: {text_val}"
        contents.append(types.Content(role=msg["role"], parts=[types.Part.from_text(text=text_val)]))

    # Eğer geçmiş boşsa sistem yönergesini doğrudan bu ilk mesaja ekle
    reply_note = f"\n\n(Reply in {LANG_NAMES.get(lang, 'English')}.)"
    if not contents:
        prompt_text = f"[System Instructions: {SYSTEM_INSTRUCTION}]\n\nUser: {latest_user_text}{reply_note}"
    else:
        prompt_text = latest_user_text + reply_note

    contents.append(types.Content(role="user", parts=[types.Part.from_text(text=prompt_text)]))
    return contents

STT_SAMPLE_RATE = 16000

def extract_answer_text(chunk) -> str:
    """Akış parçasından sadece görünür yanıt metnini al; 'thought' (düşünme) parçalarını atla."""
    texts = []
    for cand in (getattr(chunk, "candidates", None) or []):
        content = getattr(cand, "content", None)
        for part in (getattr(content, "parts", None) or []):
            if getattr(part, "thought", False):
                continue
            t = getattr(part, "text", None)
            if t:
                texts.append(t)
    return "".join(texts)

def describe_llm_chunk(chunk) -> str:
    """Boş yanıt durumunda teşhis bilgisi (finish_reason, düşünme token'ları, güvenlik engeli)."""
    if chunk is None:
        return "API hiç veri parçası döndürmedi."
    bits = []
    pf = getattr(chunk, "prompt_feedback", None)
    if pf is not None and getattr(pf, "block_reason", None):
        bits.append(f"block_reason={pf.block_reason}")
    for cand in (getattr(chunk, "candidates", None) or []):
        if getattr(cand, "finish_reason", None):
            bits.append(f"finish_reason={cand.finish_reason}")
    um = getattr(chunk, "usage_metadata", None)
    if um is not None:
        bits.append(f"thoughts_tokens={getattr(um, 'thoughts_token_count', None)}")
        bits.append(f"output_tokens={getattr(um, 'candidates_token_count', None)}")
    return ", ".join(bits) or "ayrıntı yok"

class ThoughtMarkerFilter:
    """Gemma 4 bazen düşünceyi <|channel>thought ... <channel|> işaretleriyle düz metin içinde döndürür; arada kalanı at."""
    OPEN, CLOSE = "<|channel>", "<channel|>"
    def __init__(self):
        self.raw, self.in_thought = "", False
    def feed(self, text: str) -> str:
        self.raw += text
        out = ""
        while True:
            marker = self.CLOSE if self.in_thought else self.OPEN
            idx = self.raw.find(marker)
            if idx >= 0:
                if not self.in_thought:
                    out += self.raw[:idx]
                self.raw = self.raw[idx + len(marker):]
                self.in_thought = not self.in_thought
                continue
            # İşaretin yarısı sonraki parçada gelebilir: olası öneki beklet
            keep = 0
            for k in range(min(len(marker) - 1, len(self.raw)), 0, -1):
                if marker.startswith(self.raw[-k:]):
                    keep = k
                    break
            if not self.in_thought:
                out += self.raw[:len(self.raw) - keep]
            self.raw = self.raw[len(self.raw) - keep:]
            return out
    def flush(self) -> str:
        out = "" if self.in_thought else self.raw
        self.raw = ""
        return out
# UI'nin son turdaki transkript / yanıt / hata bilgisini göstermesi için
PIPELINE_STATUS: Dict[str, str] = {"transcript": "", "response": "", "error": "", "info": ""}

PIPELINE_STATUS["stt_info"] = ""
PIPELINE_STATUS["router_info"] = ""
SUPPORTED_LANGS = ("en", "tr")

def tts_stream(cfg, text: str, lang: str):
    """Dile uygun motor/ses ile (sr, float32 numpy) parçaları üretir."""
    eng, voice = resolve_tts(cfg, lang)
    yield from REGISTRY.synthesize_tts(text, eng, voice, cfg.tts_speed)

def localized(lang: str, en: str, tr: str) -> str:
    return tr if lang == "tr" else en

def apply_vad(audio: np.ndarray, cfg: PipelineConfig):
    """
    Silero VAD (faster-whisper içinde) ile konuşma parçalarını bul, sessizlikleri at.
    Dönüş: (konuşma_sesi, konuşma_süresi_sn, parça_sayısı). VAD kapalıysa ses aynen döner.
    """
    if not cfg.stt_vad_enabled:
        return audio, len(audio) / float(STT_SAMPLE_RATE), 1
    from faster_whisper.vad import get_speech_timestamps, VadOptions
    opts = VadOptions(
        threshold=cfg.stt_vad_threshold,
        min_speech_duration_ms=cfg.stt_vad_min_speech_ms,
        min_silence_duration_ms=cfg.stt_vad_min_silence_ms,
        speech_pad_ms=cfg.stt_vad_speech_pad_ms,
    )
    stamps = get_speech_timestamps(audio, opts, sampling_rate=STT_SAMPLE_RATE)
    if not stamps:
        return audio[:0], 0.0, 0
    gap = np.zeros(int(0.1 * STT_SAMPLE_RATE), dtype=np.float32)   # parçalar arasına 100 ms sessizlik
    pieces = []
    for st in stamps:
        pieces += [audio[st["start"]:st["end"]], gap]
    speech = np.concatenate(pieces[:-1])
    return speech, len(speech) / float(STT_SAMPLE_RATE), len(stamps)

def pick_language(stt_model, speech_audio: np.ndarray, cfg: PipelineConfig):
    """
    'auto': Whisper dil olasılıklarından SADECE EN/TR arasında en yüksek olanı seç (her konuşmada yeniden).
    Böylece İngilizce başlayıp sonraki turda Türkçe'ye geçmek mümkün olur.
    İngilizce-only modellerde (distil-*, *.en) dil her zaman 'en'dir.
    """
    multilingual = getattr(getattr(stt_model, "model", None), "is_multilingual", True)
    if not multilingual:
        return "en", 1.0, cfg.stt_language != "en"
    if cfg.stt_language in SUPPORTED_LANGS:
        return cfg.stt_language, 1.0, False
    try:
        _, _, all_probs = stt_model.detect_language(speech_audio)
        probs = dict(all_probs)
        best = max(SUPPORTED_LANGS, key=lambda l: probs.get(l, 0.0))
        pe, pt = probs.get("en", 0.0), probs.get("tr", 0.0)
        return best, probs.get(best, 0.0) / max(pe + pt, 1e-9), False
    except Exception as e:
        print(f"⚠️ Dil tespiti başarısız, İngilizce varsayıldı: {e}")
        return "en", 0.0, False

def prepare_audio_for_stt(sr: int, audio_data: np.ndarray) -> np.ndarray:
    """
    Gradio mikrofon sesi (genelde 44.1/48 kHz, int16, bazen stereo) -> Whisper girişi
    (16 kHz, mono, float32 [-1, 1]). Whisper numpy dizisini HER ZAMAN 16 kHz varsayar;
    yeniden örnekleme yapılmazsa ses 3x hızlı algılanır ve transkript boş/anlamsız çıkar.
    """
    arr = np.asarray(audio_data)
    if np.issubdtype(arr.dtype, np.integer):
        arr = arr.astype(np.float32) / float(np.iinfo(arr.dtype).max + 1)   # önce normalize et
    else:
        arr = arr.astype(np.float32)
    if arr.ndim > 1:                                                         # stereo -> mono
        arr = arr.mean(axis=1) if arr.shape[1] <= arr.shape[0] else arr.mean(axis=0)
    sr = int(sr)
    if sr != STT_SAMPLE_RATE and arr.size:
        try:
            from scipy.signal import resample_poly
            g = gcd(STT_SAMPLE_RATE, sr)
            arr = resample_poly(arr, STT_SAMPLE_RATE // g, sr // g).astype(np.float32)
        except Exception:
            n_out = int(round(len(arr) * STT_SAMPLE_RATE / sr))
            arr = np.interp(np.linspace(0, len(arr) - 1, n_out), np.arange(len(arr)), arr).astype(np.float32)
    return np.clip(arr, -1.0, 1.0).astype(np.float32)

def configurable_voice_pipeline(audio: tuple[int, np.ndarray]):
    global CHAT_HISTORY
    cfg = GLOBAL_CONFIG
    pipeline_start = time.perf_counter()

    PIPELINE_STATUS.update(transcript="", response="", error="", info="", stt_info="", router_info="")

    sr, audio_data = audio
    audio_float = prepare_audio_for_stt(sr, audio_data)
    audio_in_sec = max(len(audio_float) / float(STT_SAMPLE_RATE), 0.1)

    # 1. ADIM: VAD + DİL TESPİTİ + STT & RTF ÖLÇÜMÜ
    stt_model = REGISTRY.get_stt(cfg.stt_model_id, cfg.stt_compute_type)
    t_stt = time.perf_counter()

    speech_audio, speech_sec, n_chunks = apply_vad(audio_float, cfg)
    vad_txt = (f"VAD: {speech_sec:.1f}s konuşma / {audio_in_sec:.1f}s kayıt ({n_chunks} parça)"
               if cfg.stt_vad_enabled else "VAD: kapalı")
    if speech_audio.size < int(0.2 * STT_SAMPLE_RATE):
        PIPELINE_STATUS.update(stt_info=vad_txt, info="Konuşma algılanamadı (VAD). Mikrofon seviyesini kontrol edin veya VAD eşiğini düşürün.")
        return

    lang, lang_conf, lang_forced_en = pick_language(stt_model, speech_audio, cfg)
    segments, _ = stt_model.transcribe(
        speech_audio,
        beam_size=cfg.stt_beam_size,
        language=lang,
        condition_on_previous_text=False,
        vad_filter=False          # VAD yukarıda uygulandı
    )
    user_text = " ".join([s.text for s in segments]).strip()
    stt_ms = (time.perf_counter() - t_stt) * 1000
    stt_rtf = (stt_ms / 1000.0) / audio_in_sec
    stt_info = f"{vad_txt} • Dil: {lang.upper()} ({lang_conf:.2f})"
    if lang_forced_en:
        stt_info += " • ⚠️ Seçili STT modeli sadece İngilizce; Türkçe için çok dilli model seçin"
    PIPELINE_STATUS.update(transcript=user_text, stt_info=stt_info)

    if not user_text or len(user_text) < 2:
        PIPELINE_STATUS["info"] = "Konuşma metne çevrilemedi."
        return

    # 2. ADIM: SEMANTIC INTENT ROUTER
    try:
        rt = ROUTER.route(user_text, lang, cfg)
    except Exception as e:   # router hatası konuşmayı durdurmasın: doğrudan LLM'e git
        print(f"⚠️ Intent router hatası, LLM'e yönlendiriliyor: {e}")
        rt = RouteResult(OOD_INTENT, False, 0.0, 0.0, [], None, None, 0.0, "error")
    intent, score, fast_text, cached_audio, intent_ms = rt.intent, rt.score, rt.response_text, rt.audio, rt.latency_ms
    top = ", ".join(f"{k} {v:.2f}" for k, v in rt.ranked[:2])
    PIPELINE_STATUS["router_info"] = (f"{rt.backend} → {'⚡ FAST-PATH: ' + rt.intent if rt.matched else '🧠 LLM'} "
                                      f"(skor {rt.score:.2f}, margin {rt.margin:.2f}; en yakın: {top or '-'})")
    tts_eng, tts_voice = resolve_tts(cfg, lang)
    short_profile = f"{cfg.stt_model_id.split('/')[-1][:24]} | {cfg.llm_model_id} | {tts_eng}:{tts_voice}"

    # --- FAST-PATH: Niyet yakalandıysa AI Studio API'ye hiç gitmeden RAM'den 0ms ses dön ---
    if intent != "general_llm_chat" and cfg.enable_fast_path_cache and cached_audio is not None:
        total_e2e = (time.perf_counter() - pipeline_start) * 1000
        BENCHMARK.record_turn({
            "config_summary": short_profile,
            "transcript": user_text,
            "response": fast_text,
            "intent": f"⚡ {intent} ({score:.2f}) [API BYPASSED]",
            "audio_in_sec": round(audio_in_sec, 2),
            "stt_ms": round(stt_ms, 1),
            "stt_rtf": round(stt_rtf, 4),
            "intent_ms": round(intent_ms, 1),
            "lang": lang, "speech_sec": round(speech_sec, 2),
            "llm_ttft_ms": 0.0,
            "llm_tps": 0.0,
            "tts_first_ms": 0.4,
            "tts_rtf": 0.0001,
            "total_e2e_ms": round(total_e2e, 1)
        })
        CHAT_HISTORY.append({"role": "user", "text": user_text})
        CHAT_HISTORY.append({"role": "model", "text": fast_text})
        PIPELINE_STATUS.update(response=fast_text, info=f"⚡ Fast-path: {intent} ({score:.2f})")
        yield cached_audio
        return

    # --- SLOW-PATH: Google AI Studio Gemma Streaming API + Cümle Bölücü + TTS ---
    try:
        client = REGISTRY.get_genai_client(cfg.gemini_api_key)
    except Exception as e:
        err_msg = localized(lang, "Please enter your Google AI Studio API key in the configuration panel.",
                            "Lütfen yapılandırma panelinden Google AI Studio API anahtarınızı girin.")
        PIPELINE_STATUS["error"] = str(e)
        for out_sr, audio_chunk in tts_stream(cfg, err_msg, lang):
            yield (out_sr, audio_chunk)
        return

    gemma_contents = build_gemma_contents(CHAT_HISTORY, user_text, lang)

    t_llm_start = time.perf_counter()

    buffer = ""
    full_response = ""
    approx_tokens = 0
    first_token_ms = None
    first_audio_yielded = False
    tts_first_ms = 0.0
    total_e2e_ms = 0.0
    total_tts_compute_sec = 0.0
    total_tts_audio_sec = 0.0
    api_error = None

    try:
        response_stream = client.models.generate_content_stream(
            model=cfg.llm_model_id,
            contents=gemma_contents,
            config=types.GenerateContentConfig(
                temperature=cfg.llm_temperature,
                max_output_tokens=cfg.llm_max_tokens + cfg.llm_thinking_reserve,
            )
        )
        thought_filter = ThoughtMarkerFilter()
        last_chunk = None

        # NOT: generate_content_stream bir generator döndürür; 401/403/404/429 gibi API hataları
        # çağrıda değil, ilk chunk okunurken fırlar -> iterasyon da try bloğunun içinde olmalı.
        for chunk in response_stream:
            last_chunk = chunk
            delta = thought_filter.feed(extract_answer_text(chunk))
            if not delta:
                continue

            if first_token_ms is None:
                first_token_ms = (time.perf_counter() - t_llm_start) * 1000

            # Kelime/karakter bazlı yaklaşık token hesabı (1 token ≈ 4 karakter)
            approx_tokens += max(len(delta) / 4.0, 1.0)
            buffer += delta
            full_response += delta

            # Akıllı Cümle Bölücü (Sentence Chunker)
            if re.search(r'[.!?,;]+(\s|$)', buffer) and len(buffer.split()) >= cfg.chunk_min_words:
                sentence = buffer.strip().replace("*", "")
                buffer = ""

                t_tts = time.perf_counter()
                for out_sr, audio_chunk in tts_stream(cfg, sentence, lang):
                    chunk_comp_sec = time.perf_counter() - t_tts
                    total_tts_compute_sec += chunk_comp_sec
                    total_tts_audio_sec += len(audio_chunk) / float(out_sr)

                    if not first_audio_yielded:
                        tts_first_ms = chunk_comp_sec * 1000
                        total_e2e_ms = (time.perf_counter() - pipeline_start) * 1000
                        first_audio_yielded = True
                    yield (out_sr, audio_chunk)
                    t_tts = time.perf_counter()

        tail = thought_filter.flush()
        buffer += tail
        full_response += tail
        if not full_response.strip():
            diag = describe_llm_chunk(last_chunk)
            PIPELINE_STATUS["error"] = f"{cfg.llm_model_id} boş yanıt döndürdü ({diag}). Başka bir model deneyin veya max token/thinking bütçesini artırın."
            print(f"⚠️ Boş LLM yanıtı: {diag}")
            for out_sr, audio_chunk in tts_stream(cfg, localized(lang, "The model returned an empty response.", "Model boş bir yanıt döndürdü."), lang):
                yield (out_sr, audio_chunk)
    except Exception as api_err:
        api_error = api_err
        print(f"❌ AI Studio API Hatası ({cfg.llm_model_id}): {api_err}")
        PIPELINE_STATUS["error"] = f"{cfg.llm_model_id}: {api_err}"
        if not first_audio_yielded:
            err_msg = localized(lang, f"API error occurred with model {cfg.llm_model_id}.",
                                f"{cfg.llm_model_id} modelinde bir API hatası oluştu.")
            for out_sr, audio_chunk in tts_stream(cfg, err_msg, lang):
                yield (out_sr, audio_chunk)
        buffer = ""   # yarım kalan cümleyi sentezlemeye çalışma

    llm_net_sec = max((time.perf_counter() - t_llm_start) - total_tts_compute_sec, 0.01)
    llm_tps = approx_tokens / llm_net_sec

    if buffer.strip():
        clean_rem = buffer.strip().replace("*", "")
        t_tts = time.perf_counter()
        for out_sr, audio_chunk in tts_stream(cfg, clean_rem, lang):
            chunk_comp_sec = time.perf_counter() - t_tts
            total_tts_compute_sec += chunk_comp_sec
            total_tts_audio_sec += len(audio_chunk) / float(out_sr)
            if not first_audio_yielded:
                tts_first_ms = chunk_comp_sec * 1000
                total_e2e_ms = (time.perf_counter() - pipeline_start) * 1000
                first_audio_yielded = True
            yield (out_sr, audio_chunk)
            t_tts = time.perf_counter()

    if not first_audio_yielded:
        total_e2e_ms = (time.perf_counter() - pipeline_start) * 1000
    tts_rtf = total_tts_compute_sec / max(total_tts_audio_sec, 0.1)

    BENCHMARK.record_turn({
        "config_summary": short_profile,
        "transcript": user_text,
        "response": full_response,
        "intent": f"🧠 {intent} ({score:.2f})" + (" [API ERROR]" if api_error else ""),
        "audio_in_sec": round(audio_in_sec, 2),
        "stt_ms": round(stt_ms, 1),
        "stt_rtf": round(stt_rtf, 4),
        "intent_ms": round(intent_ms, 1),
        "lang": lang, "speech_sec": round(speech_sec, 2),
        "llm_ttft_ms": round(first_token_ms or 0.0, 1),
        "llm_tps": round(llm_tps, 1),
        "tts_first_ms": round(tts_first_ms, 1),
        "tts_rtf": round(tts_rtf, 4),
        "total_e2e_ms": round(total_e2e_ms, 1)
    })
    PIPELINE_STATUS["response"] = full_response
    if api_error is None and full_response.strip():
        CHAT_HISTORY.append({"role": "user", "text": user_text})
        CHAT_HISTORY.append({"role": "model", "text": full_response})

# Hücre 5: AI Studio Gemma Model Seçicili WebRTC Vitrin & Benchmark Arayüzü

Bu hücrede "🔄 AI Studio'dan Gemma Modellerini Listele" butonu yer alır. API anahtarınızı girdiğinizde hesabınıza tanımlı tüm ücretsiz Gemma modellerini (gemma-3-27b-it, gemma-3-12b-it, gemma-3-4b-it, gemma-3-1b-it vb.) canlı olarak açılır menüye (Dropdown) doldurur ve her modelin gecikmesini yan paneldeki tabloda kıyaslamanızı sağlar.

In [ ]:
import inspect
import gradio as gr
import numpy as np
import pandas as pd

# Başlangıçta mevcut anahtarla (varsa) Gemma listesini çek
INITIAL_GEMMA_LIST = fetch_aistudio_gemma_models(GLOBAL_CONFIG.gemini_api_key)
if GLOBAL_CONFIG.llm_model_id not in INITIAL_GEMMA_LIST and INITIAL_GEMMA_LIST:
    GLOBAL_CONFIG.llm_model_id = INITIAL_GEMMA_LIST[0]

def refresh_gemma_dropdown(api_key: str):
    GLOBAL_CONFIG.gemini_api_key = api_key.strip()
    models = fetch_aistudio_gemma_models(api_key)
    selected = models[0] if models else "gemma-3-27b-it"
    return gr.update(choices=models, value=selected), f"✅ AI Studio'dan **{len(models)} adet Gemma modeli** başarıyla listelendi!"

def on_engine_change(engine):
    """TTS motoru değişince İngilizce ses listesini motora göre güncelle (Kokoro/Piper profilleri karışmasın)."""
    if engine == "kokoro":
        choices, value = KOKORO_VOICES, GLOBAL_CONFIG.tts_voice
    else:
        choices, value = piper_voice_choices("en"), GLOBAL_CONFIG.piper_voice_en
    if value not in choices:
        value = choices[0]
    label = "Kokoro Ses Profili (İngilizce)" if engine == "kokoro" else "Piper Ses Profili (İngilizce)"
    return gr.update(choices=choices, value=value, label=label)

def backend_info_md(key: str) -> str:
    spec = INTENT_BACKENDS.get(key)
    if spec is None:
        return "*Bilinmeyen motor*"
    return (f"**Lisans:** `{spec.license or '-'}` • **Önerilen başlangıç eşiği:** `{spec.default_threshold}` "
            f"• **margin:** `{spec.default_margin}`\n\n{spec.description}")

def on_backend_change(key):
    spec = INTENT_BACKENDS[key]
    return backend_info_md(key), gr.update(value=spec.default_threshold), gr.update(value=spec.default_margin)

def router_test(text, lang):
    if not (text or "").strip():
        return "⚠️ Bir cümle yazın.", pd.DataFrame(columns=["Intent", "Skor"])
    if ROUTER.backend is None:
        return "⚠️ Router etkin değil. Önce 'Konfigürasyonu Uygula'ya basın.", pd.DataFrame(columns=["Intent", "Skor"])
    ranked = ROUTER.rank(text, lang)
    intent, ok, s1, margin = ROUTER.decide(ranked, GLOBAL_CONFIG.intent_threshold, GLOBAL_CONFIG.intent_margin)
    verdict = (f"⚡ **FAST-PATH → `{intent}`**" if ok else "🧠 **LLM'e gider** (eşik/margin sağlanmadı)")
    md = (f"{verdict}\n\nMotor: `{ROUTER.backend_key}` • skor `{s1:.3f}` (eşik `{GLOBAL_CONFIG.intent_threshold}`) "
          f"• margin `{margin:.3f}` (min `{GLOBAL_CONFIG.intent_margin}`) • normalize: `{normalize_text(text, lang)}`")
    return md, pd.DataFrame([{"Intent": k, "Skor": round(v, 3)} for k, v in ranked])

def router_calibrate():
    if ROUTER.backend is None:
        return "⚠️ Router etkin değil. Önce 'Konfigürasyonu Uygula'ya basın.", gr.update()
    best, table, ranked_all = evaluate_intent_router(ROUTER, INTENT_TEST_SET, GLOBAL_CONFIG.intent_margin)
    thr = best["threshold"]
    errs = []
    for ranked, (t, l, exp) in zip(ranked_all, INTENT_TEST_SET):
        pred, ok, s1, _ = ROUTER.decide(ranked, thr, GLOBAL_CONFIG.intent_margin)
        if pred != exp:
            errs.append(f"- `{t}` ({l}) → **{pred}** (skor {s1:.2f}), beklenen: `{exp}`")
    md = (f"### 🎯 Kalibrasyon — `{ROUTER.backend_key}` ({len(INTENT_TEST_SET)} test cümlesi)\n"
          f"**Önerilen eşik: `{thr}`** → doğruluk %{best['accuracy']*100:.0f}, yanlış fast-path %{best['wrong_fast']*100:.0f}, "
          f"LLM'e düşen doğru intent %{best['missed_to_llm']*100:.0f}\n\n"
          + "| Eşik | Doğruluk | Yanlış fast-path | LLM'e düşen doğru intent |\n| :-- | :-- | :-- | :-- |\n"
          + "\n".join(f"| {r.threshold} | %{r.accuracy*100:.0f} | %{r.wrong_fast*100:.0f} | %{r.missed_to_llm*100:.0f} |" for r in table.itertuples())
          + "\n\n**Hatalar:**\n" + ("\n".join(errs) if errs else "yok 🎉")
          + "\n\n*Slider önerilen değere çekildi; 'Konfigürasyonu Uygula' ile kalıcı yapın.*")
    return md, gr.update(value=thr)

def apply_configuration(
    api_key, gemma_model, llm_temp, chunk_words,
    stt_model, stt_compute, stt_beam, stt_lang,
    vad_on, vad_thr, vad_min_speech, vad_min_silence, vad_pad,
    intent_backend, intent_thresh, intent_margin, fast_cache,
    tts_engine, tts_voice_en, tts_voice_tr, tts_speed
):
    GLOBAL_CONFIG.gemini_api_key = api_key.strip()
    GLOBAL_CONFIG.llm_model_id = gemma_model
    GLOBAL_CONFIG.llm_temperature = float(llm_temp)
    GLOBAL_CONFIG.chunk_min_words = int(chunk_words)

    GLOBAL_CONFIG.stt_model_id = stt_model
    GLOBAL_CONFIG.stt_compute_type = stt_compute
    GLOBAL_CONFIG.stt_beam_size = int(stt_beam)
    GLOBAL_CONFIG.stt_language = stt_lang

    GLOBAL_CONFIG.stt_vad_enabled = bool(vad_on)
    GLOBAL_CONFIG.stt_vad_threshold = float(vad_thr)
    GLOBAL_CONFIG.stt_vad_min_speech_ms = int(vad_min_speech)
    GLOBAL_CONFIG.stt_vad_min_silence_ms = int(vad_min_silence)
    GLOBAL_CONFIG.stt_vad_speech_pad_ms = int(vad_pad)

    GLOBAL_CONFIG.intent_backend = intent_backend
    GLOBAL_CONFIG.intent_threshold = float(intent_thresh)
    GLOBAL_CONFIG.intent_margin = float(intent_margin)
    GLOBAL_CONFIG.enable_fast_path_cache = bool(fast_cache)

    GLOBAL_CONFIG.tts_engine = tts_engine
    if tts_engine == "kokoro":
        GLOBAL_CONFIG.tts_voice = tts_voice_en
    else:
        GLOBAL_CONFIG.piper_voice_en = tts_voice_en
    GLOBAL_CONFIG.piper_voice_tr = tts_voice_tr
    GLOBAL_CONFIG.tts_speed = float(tts_speed)

    # Yerel modelleri ve fast-path ses önbelleğini ısıt (hem İngilizce hem Türkçe ses)
    try:
        REGISTRY.get_stt(stt_model, stt_compute)
        for lang, txt in (("en", "Ready."), ("tr", "Hazır.")):
            eng, voice = resolve_tts(GLOBAL_CONFIG, lang)
            _ = list(REGISTRY.synthesize_tts(txt, eng, voice, float(tts_speed)))
        eng_en, voice_en = resolve_tts(GLOBAL_CONFIG, "en")
        ROUTER.activate(intent_backend)
        ROUTER.prepare_audio(GLOBAL_CONFIG)
    except Exception as e:
        return f"❌ **Konfigürasyon uygulanamadı:** `{type(e).__name__}: {e}`"

    vram = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    lang_label = {"auto": "EN+TR otomatik", "en": "EN", "tr": "TR"}.get(stt_lang, stt_lang)
    vad_label = (f"VAD eşik {float(vad_thr):.2f}" if vad_on else "VAD kapalı")
    return (f"✅ **Aktif Profil Güncellendi:** `{gemma_model}` + `{stt_model.split('/')[-1]}` ({lang_label}, {vad_label}) + Router `{intent_backend}` "
            f"+ `{tts_engine.upper()}` EN:`{voice_en}` TR:`{GLOBAL_CONFIG.piper_voice_tr}` (Yerel VRAM: `{vram:.2f} GB`)")

def refresh_dashboard():
    t = BENCHMARK.latest
    stats = BENCHMARK.get_percentiles()
    ram_pct = psutil.virtual_memory().percent

    hud_md = f"""
    ### ⚡ Canlı Konuşma Telemetrisi (Turn #{t['turn_id']}) — Aktif Profil: `{t['config_summary']}`
    | Katman | Gecikme (Latency) | Verimlilik / Hız (RTF & Throughput) | Hedef Eşik |
    | :--- | :--- | :--- | :--- |
    | **1. STT ({GLOBAL_CONFIG.stt_compute_type}, Beam={GLOBAL_CONFIG.stt_beam_size})** | `{t['stt_ms']} ms` | **RTF:** `{t['stt_rtf']}x` *(Ses: {t['audio_in_sec']}s)* | `RTF < 0.15x` |
    | **   ↳ Dil / VAD** | `{t.get('lang', '-')}` | Konuşma: `{t.get('speech_sec', 0.0)} s` / Kayıt: `{t['audio_in_sec']} s` | VAD: `{'açık' if GLOBAL_CONFIG.stt_vad_enabled else 'kapalı'}` |
    | **2. Intent Router** (`{GLOBAL_CONFIG.intent_backend}`) | `{t['intent_ms']} ms` | `{t['intent']}` | `< 25 ms` |
    | **3. AI Studio ({GLOBAL_CONFIG.llm_model_id})** | **TTFT:** `{t['llm_ttft_ms']} ms` | **Akış Hızı:** `~{t['llm_tps']} tokens/sec` | `TTFT < 350 ms` |
    | **4. TTS ({GLOBAL_CONFIG.tts_engine.upper()})** | **İlk Ses:** `{t['tts_first_ms']} ms` | **RTF:** `{t['tts_rtf']}x` | `RTF < 0.25x` |
    | **🔥 TOPLAM UÇTAN UCA (E2E)** | **`{t['total_e2e_ms']} ms`** | **P50:** `{stats['p50_e2e']} ms` • **P95:** `{stats['p95_e2e']} ms` | **`< 650 ms`** |

    **🖥️ Sistem & Maliyet:** `Yerel GPU VRAM: {t['vram_gb']} GB / 15.0 GB` • `RAM: %{ram_pct}` • `LLM Maliyeti: $0.00 (AI Studio Free Tier)`
    **🗣️ Kullanıcı:** *{t['transcript']}*
    **🤖 Gemma Yanıtı:** *{t['response']}*
    """
    return hud_md, BENCHMARK.get_summary_df()

def process_audio_fallback(audio):
    """Mikrofon/yükleme sesini işler -> (yanıt sesi, durum mesajı). Hatalar arayüzde gösterilir."""
    if audio is None:
        return None, "⚠️ Ses alınamadı. Kaydı durdurduktan sonra birkaç saniye bekleyip tekrar deneyin."

    out_sr = 24000
    out_chunks = []
    try:
        for sr, chunk in configurable_voice_pipeline(audio):
            out_sr = sr
            out_chunks.append(np.asarray(chunk, dtype=np.float32).reshape(-1))
    except Exception as e:
        import traceback
        traceback.print_exc()
        return None, f"❌ **Pipeline hatası:** `{type(e).__name__}: {e}`"

    st = PIPELINE_STATUS
    lines = []
    if st["transcript"]:
        lines.append(f"🗣️ **Algılanan:** {st['transcript']}")
    if st["response"]:
        lines.append(f"🤖 **Yanıt:** {st['response']}")
    if st.get("stt_info"):
        lines.append(f"🎚️ {st['stt_info']}")
    if st.get("router_info"):
        lines.append(f"🧭 {st['router_info']}")
    if st["info"]:
        lines.append(f"ℹ️ {st['info']}")
    if st["error"]:
        lines.append(f"❌ **API hatası:** `{st['error']}`")
    status_md = "\n\n".join(lines) or "⚠️ Yanıt üretilemedi."

    if not out_chunks:
        return None, status_md

    final_audio = np.clip(np.concatenate(out_chunks), -1.0, 1.0)
    # Gradio'ya açıkça int16 PCM ver (float -> int16 otomatik dönüşüm uyarısı/taşması olmasın)
    return (out_sr, (final_audio * 32767).astype(np.int16)), status_md

# Gradio 6'da 'theme' Blocks() yerine launch()'a taşındı; 5.x'te tam tersi. İkisiyle de çalışsın:
_THEME_IN_LAUNCH = "theme" in inspect.signature(gr.Blocks.launch).parameters
_blocks_kwargs = {} if _THEME_IN_LAUNCH else {"theme": gr.themes.Soft()}

with gr.Blocks(**_blocks_kwargs) as demo:
    gr.Markdown(
        "# 🎙️ Voice AI Workbench (Standart HTTP Modu)\n"
        "*WebRTC devre dışı bırakıldı. Lütfen mikrofona tıklayıp konuşun ve kaydı durdurun.*"
    )

    with gr.Row():
        # SOL PANEL
        with gr.Column(scale=5):
            with gr.Group():
                audio_in = gr.Audio(sources=["microphone", "upload"], type="numpy", label="Sesi Kaydet (Konuşmayı bitirince durdurun)")
                process_btn = gr.Button("▶️ Kaydı İşle (otomatik tetiklenmezse)", variant="secondary")
                audio_out = gr.Audio(label="Yapay Zeka Yanıtı", autoplay=True)
                turn_status = gr.Markdown("*Mikrofona tıklayıp konuşun, sonra kaydı durdurun.*")

            # Kayıt bitince veya dosya yüklenince otomatik çalıştır; buton manuel yedek tetikleyici
            audio_in.stop_recording(fn=process_audio_fallback, inputs=[audio_in], outputs=[audio_out, turn_status])
            audio_in.upload(fn=process_audio_fallback, inputs=[audio_in], outputs=[audio_out, turn_status])
            process_btn.click(fn=process_audio_fallback, inputs=[audio_in], outputs=[audio_out, turn_status])

            with gr.Accordion("⚙️ Model Seçimi & Hiper-Parametre Konfigürasyonu", open=True):
                with gr.Tab("1. AI Studio (Gemma API)"):
                    api_key_tb = gr.Textbox(
                        value=GLOBAL_CONFIG.gemini_api_key,
                        type="password",
                        label="Google AI Studio API Key (GEMINI_API_KEY)",
                        placeholder="AIzaSy..."
                    )
                    with gr.Row():
                        gemma_dd = gr.Dropdown(
                            choices=INITIAL_GEMMA_LIST,
                            value=GLOBAL_CONFIG.llm_model_id,
                            label="Ücretsiz Gemma Modelleri (AI Studio)"
                        )
                        fetch_btn = gr.Button("🔄 Gemma Modellerini Listele", scale=0)
                    with gr.Row():
                        llm_temp_sl = gr.Slider(0.0, 1.0, value=GLOBAL_CONFIG.llm_temperature, step=0.1, label="Temperature")
                        chunk_sl = gr.Slider(2, 12, value=4, step=1, label="Cümle Bölücü Min. Kelime (Chunker)")

                with gr.Tab("2. STT & VAD"):
                    stt_model_dd = gr.Dropdown(
                        choices=[
                            ("Whisper large-v3-turbo — çok dilli (TR+EN), hızlı", "mobiuslabsgmbh/faster-whisper-large-v3-turbo"),
                            ("Whisper large-v3 — çok dilli (TR+EN), en doğru", "Systran/faster-whisper-large-v3"),
                            ("Whisper medium — çok dilli (TR+EN)", "Systran/faster-whisper-medium"),
                            ("Whisper small — çok dilli (TR+EN), çok hızlı", "Systran/faster-whisper-small"),
                            ("Distil large-v3 — SADECE İngilizce", "Systran/faster-distil-whisper-large-v3"),
                            ("Distil small.en — SADECE İngilizce", "Systran/faster-distil-whisper-small.en"),
                        ],
                        value=GLOBAL_CONFIG.stt_model_id, label="Yerel STT Modeli (CUDA) — Türkçe için çok dilli model seçin"
                    )
                    stt_lang_dd = gr.Dropdown(
                        choices=[("Otomatik (her konuşmada İngilizce/Türkçe seçilir)", "auto"), ("Sadece İngilizce", "en"), ("Sadece Türkçe", "tr")],
                        value=GLOBAL_CONFIG.stt_language, label="Konuşma Dili (STT + yanıt dili)"
                    )
                    with gr.Row():
                        stt_comp_dd = gr.Dropdown(["int8_float16", "float16", "int8"], value=GLOBAL_CONFIG.stt_compute_type, label="Quantization")
                        stt_beam_sl = gr.Slider(1, 5, value=GLOBAL_CONFIG.stt_beam_size, step=1, label="Beam Size")

                    gr.Markdown("**🎚️ VAD (Voice Activity Detection — Silero):** kaydın sessiz/gürültülü kısımlarını atar; konuşma yoksa STT hiç çalışmaz.")
                    vad_on_cb = gr.Checkbox(value=GLOBAL_CONFIG.stt_vad_enabled, label="VAD aktif")
                    with gr.Row():
                        vad_thr_sl = gr.Slider(0.1, 0.9, value=GLOBAL_CONFIG.stt_vad_threshold, step=0.05, label="VAD Eşiği (yüksek = daha seçici)")
                        vad_speech_sl = gr.Slider(50, 1000, value=GLOBAL_CONFIG.stt_vad_min_speech_ms, step=50, label="Min. Konuşma (ms)")
                    with gr.Row():
                        vad_silence_sl = gr.Slider(100, 2000, value=GLOBAL_CONFIG.stt_vad_min_silence_ms, step=100, label="Min. Sessizlik (ms)")
                        vad_pad_sl = gr.Slider(0, 800, value=GLOBAL_CONFIG.stt_vad_speech_pad_ms, step=50, label="Konuşma Tamponu (ms)")

                with gr.Tab("3. TTS Sentezleyici"):
                    with gr.Row():
                        tts_eng_dd = gr.Dropdown(
                            ["piper", "kokoro"], value=GLOBAL_CONFIG.tts_engine,
                            label="TTS Motoru (Piper = CPU, EN+TR • Kokoro = GPU, sadece EN)"
                        )
                        tts_voice_dd = gr.Dropdown(
                            choices=(KOKORO_VOICES if GLOBAL_CONFIG.tts_engine == "kokoro" else piper_voice_choices("en")),
                            value=(GLOBAL_CONFIG.tts_voice if GLOBAL_CONFIG.tts_engine == "kokoro" else GLOBAL_CONFIG.piper_voice_en),
                            allow_custom_value=True,   # choices motora göre dinamik değişiyor; sunucu tarafı doğrulaması takılmasın
                            label=("Kokoro Ses Profili (İngilizce)" if GLOBAL_CONFIG.tts_engine == "kokoro" else "Piper Ses Profili (İngilizce)")
                        )
                    tts_voice_tr_dd = gr.Dropdown(
                        choices=piper_voice_choices("tr"), value=GLOBAL_CONFIG.piper_voice_tr,
                        label="Piper Ses Profili (Türkçe) — Türkçe yanıtlar her zaman Piper ile seslendirilir"
                    )
                    tts_speed_sl = gr.Slider(0.8, 1.4, value=GLOBAL_CONFIG.tts_speed, step=0.05, label="Konuşma Hızı")
                    tts_eng_dd.change(fn=on_engine_change, inputs=[tts_eng_dd], outputs=[tts_voice_dd])

                with gr.Tab("4. Intent Router (TR+EN)"):
                    backend_dd = gr.Dropdown(
                        choices=[(s.label, k) for k, s in INTENT_BACKENDS.items()],
                        value=GLOBAL_CONFIG.intent_backend, label="Intent Router Motoru (hepsi açık kaynak, çok dilli)"
                    )
                    backend_info = gr.Markdown(backend_info_md(GLOBAL_CONFIG.intent_backend))
                    with gr.Row():
                        intent_sl = gr.Slider(0.10, 0.95, value=GLOBAL_CONFIG.intent_threshold, step=0.01, label="Intent Eşik Değeri (Threshold)")
                        intent_margin_sl = gr.Slider(0.0, 0.30, value=GLOBAL_CONFIG.intent_margin, step=0.01, label="Min. Margin (1. - 2. intent farkı)")
                    fast_cache_cb = gr.Checkbox(value=GLOBAL_CONFIG.enable_fast_path_cache, label="Fast-Path RAM Audio Cache (API Bypass)")

                    gr.Markdown("**🧪 Router Test Alanı** — sesli konuşmadan önce cümleleri buradan deneyin (aktif motor + güncel eşik).")
                    with gr.Row():
                        rt_text = gr.Textbox(label="Test cümlesi", placeholder="Kargom ne zaman gelir?", scale=4)
                        rt_lang = gr.Radio([("Türkçe", "tr"), ("İngilizce", "en")], value="tr", label="Dil", scale=1)
                    rt_btn = gr.Button("🔍 Router'ı Test Et")
                    rt_md = gr.Markdown()
                    rt_df = gr.Dataframe(interactive=False, label="Intent skorları")
                    calib_btn = gr.Button("🎯 Test Setiyle Eşiği Kalibre Et")
                    calib_md = gr.Markdown()

                    backend_dd.change(fn=on_backend_change, inputs=[backend_dd], outputs=[backend_info, intent_sl, intent_margin_sl])
                    rt_btn.click(fn=router_test, inputs=[rt_text, rt_lang], outputs=[rt_md, rt_df])
                    rt_text.submit(fn=router_test, inputs=[rt_text, rt_lang], outputs=[rt_md, rt_df])
                    calib_btn.click(fn=router_calibrate, inputs=[], outputs=[calib_md, intent_sl])

                apply_btn = gr.Button("🚀 Konfigürasyonu Uygula", variant="primary")
                config_status = gr.Markdown("*Varsayılan profil aktif.*")

        # SAĞ PANEL
        with gr.Column(scale=7):
            hud_display = gr.Markdown()
            gr.Markdown("### 📈 Gemma Modelleri & Pipeline Kıyaslama Tablosu (Son 10 Konuşma)")
            bench_table = gr.Dataframe(interactive=False)

            timer = gr.Timer(1.0)
            timer.tick(fn=refresh_dashboard, outputs=[hud_display, bench_table])

    # Event bağlantıları
    fetch_btn.click(
        fn=refresh_gemma_dropdown,
        inputs=[api_key_tb],
        outputs=[gemma_dd, config_status]
    )

    apply_btn.click(
        fn=apply_configuration,
        inputs=[
            api_key_tb, gemma_dd, llm_temp_sl, chunk_sl,
            stt_model_dd, stt_comp_dd, stt_beam_sl, stt_lang_dd,
            vad_on_cb, vad_thr_sl, vad_speech_sl, vad_silence_sl, vad_pad_sl,
            backend_dd, intent_sl, intent_margin_sl, fast_cache_cb,
            tts_eng_dd, tts_voice_dd, tts_voice_tr_dd, tts_speed_sl
        ],
        outputs=config_status
    )

_launch_kwargs = {"theme": gr.themes.Soft()} if _THEME_IN_LAUNCH else {}
demo.queue().launch(share=True, debug=True, **_launch_kwargs)
